# Experiment 08: Independent Sublayer DBSCAN Tucker Compression

**Target Model**: `google/gemma-3-1b-it` (Layer 0 `model.layers[0]`)  
**Target Submodules**: All 3 MLP Projections (`gate_proj`, `up_proj`, and `down_proj`)  
**Evaluation Task**: GLUE MNLI Validation Set (`validation_matched`)  

### Core Architecture & Innovations:
1. **Dedicated Independent Sublayer Profiling**:
   Rather than imposing `gate_proj`'s activation clusters onto `up_proj` and `down_proj`, each submodule is independently profiled during baseline inference:
   - `act_gate`: Captured at `act_fn` output ($[B, S, 6912]$, non-linear gating distribution).
   - `act_up`: Captured at `up_proj` output ($[B, S, 6912]$, linear feature magnitude distribution).
   - `act_down`: Captured at `down_proj` input ($[B, S, 6912]$, compound element-wise product distribution).
2. **Dedicated DBSCAN Clustering**:
   Each submodule independently runs DBSCAN with calibrated $\epsilon$ and min density, discovering its own natural density-connected operational regimes.
3. **Independent Superweight Quarantine**:
   Isolated noise coordinates (DBSCAN label `-1`, $|x| > 3.0$ or top 1% variance) are quarantined in uncompressed FP32 for each submodule independently.
4. **Multi-Tier Comparative Sweeps**:
   - **Moderate Sweet-Spot Sweep (`[4, 180, 600]` on all 3)**: Calibrated to ~76–78% reconstruction error across all submodules.
   - **Balanced Sweep (`[4, 250, 800]` on all 3)**: Lower reconstruction error (~68–70%).
   - **Heterogeneous / Mixed Optimal Tier**:
     `gate_proj`: `[3, 100, 350]` (aggressive regularizing cut)  
     `up_proj`: `[4, 180, 600]` (moderate high-fidelity cut)  
     `down_proj`: `[4, 180, 600]` (moderate high-fidelity cut)  
   - **Aggressive Sweep (`[3, 100, 350]` on all 3)**: Directly compares independent vs. coupled clustering at the aggressive rank.
5. **Optimization**:
   PyTorch Adam Gradient Descent (35 steps, $lr=10^{-3}$) refines each submodule's Tucker core and factors.

In [1]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Library Imports
# =====================================================================
import os
import sys
import time
from pathlib import Path
import math
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.cluster import DBSCAN
from sklearn.metrics import accuracy_score
from IPython import get_ipython

# Set TensorLy PyTorch backend
tl.set_backend("pytorch")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Minimal Time Logging
GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

# Neural Decomp framework imports
try:
    from neural_decomp import ModelManagementInterface, DeviceMapOptions
    from neural_decomp.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from neural_decomp.utils import get_device_info, save_json_metrics
    print("Loaded neural_decomp library.")
except ImportError:
    from utility import ModelManagementInterface, DeviceMapOptions
    from utility.decomposition import decompose_svd, truncate_svd, decompose_tucker
    from utility.utils import get_device_info, save_json_metrics
    print("Loaded utility library.")

device_info = get_device_info()
print(f"Device: {device_info['device_name']} | CUDA Available: {device_info['cuda_available']}")

/home/dwithun/Development/llm_compression/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded neural_decomp library.
Device: NVIDIA GeForce RTX 3070 Ti | CUDA Available: True


In [2]:
# =====================================================================
# STEP 2: Initialize Model & Tokenizer on GPU (Unprocessed Baseline)
# =====================================================================
model_id = "google/gemma-3-1b-it"

mmi = ModelManagementInterface(
    model_id=model_id,
    precision=torch.float32,
    device_map=DeviceMapOptions.AUTO,
)
model = mmi.get_model()
tokenizer = mmi.get_tokenizer()

target_layer = model.model.layers[0]
D_IN = target_layer.mlp.gate_proj.weight.shape[1]
D_OUT = target_layer.mlp.gate_proj.weight.shape[0]

print(f"Target Feedforward Layer 0: in_features={D_IN}, intermediate_features={D_OUT}")

# Cache pristine Layer 0 weights on CPU
W_gate_orig = target_layer.mlp.gate_proj.weight.data.clone().cpu()
W_up_orig   = target_layer.mlp.up_proj.weight.data.clone().cpu()
W_down_orig = target_layer.mlp.down_proj.weight.data.clone().cpu()
print("Cached pristine Layer 0 weights on CPU.")

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 369.25it/s]


Target Feedforward Layer 0: in_features=1152, intermediate_features=6912
Cached pristine Layer 0 weights on CPU.
time: 4.35s
cummulative_time: 6.16s


In [3]:
# =====================================================================
# STEP 3: Load GLUE MNLI Validation Benchmark
# =====================================================================
ds = load_dataset("nyu-mll/glue", "mnli")["validation_matched"]

label_names = ["entailment", "neutral", "contradiction"]
label_token_ids = [tokenizer.encode(" " + name, add_special_tokens=False)[0] for name in label_names]

EVAL_SAMPLE_COUNT = 1000
eval_data = ds.select(range(EVAL_SAMPLE_COUNT))

print(f"Loaded GLUE MNLI: {len(ds):,} total samples | Active Evaluation Subset: {len(eval_data):,} samples")

Loaded GLUE MNLI: 9,815 total samples | Active Evaluation Subset: 1,000 samples
time: 3.49s
cummulative_time: 9.65s


## Step 1: Multi-Hook Activation Profiling on Baseline Weights

We register simultaneous forward hooks on all 3 submodules of Layer 0:
1. `hook_gate`: On `mlp.act_fn` (captures non-linear gating signals).
2. `hook_up`: On `mlp.up_proj` (captures linear feature magnitude signals).
3. `hook_down`: On `mlp.down_proj` input (captures the compound element-wise product).

In [4]:
# =====================================================================
# STEP 4: Baseline MNLI Inference & Independent Activation Profiling
# =====================================================================
acts_gate, acts_up, acts_down = [], [], []

def hook_gate_fn(module, input_tensor, output_tensor):
    act = output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor
    acts_gate.append(act.detach().cpu().squeeze(0).mean(dim=0).numpy())

def hook_up_fn(module, input_tensor, output_tensor):
    act = output_tensor[0] if isinstance(output_tensor, tuple) else output_tensor
    acts_up.append(act.detach().cpu().squeeze(0).mean(dim=0).numpy())

def hook_down_fn(module, input_tensor, output_tensor):
    inp = input_tensor[0] if isinstance(input_tensor, tuple) else input_tensor
    acts_down.append(inp.detach().cpu().squeeze(0).mean(dim=0).numpy())

h1 = target_layer.mlp.act_fn.register_forward_hook(hook_gate_fn)
h2 = target_layer.mlp.up_proj.register_forward_hook(hook_up_fn)
h3 = target_layer.mlp.down_proj.register_forward_hook(hook_down_fn)

predictions, ground_truth = [], []

model.eval()
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Baseline Profiling & MNLI Inference"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs, logits_to_keep=1)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        predictions.append(pred_label)
        ground_truth.append(sample["label"])

h1.remove()
h2.remove()
h3.remove()

M_gate = np.stack(acts_gate)
M_up = np.stack(acts_up)
M_down = np.stack(acts_down)

baseline_accuracy = accuracy_score(ground_truth, predictions)

print(f"\nUncompressed Baseline Accuracy: {baseline_accuracy * 100:.2f}%")
print(f"Captured Activation Profiles:")
print(f"  gate_proj (act_fn output):   shape {M_gate.shape} | mean={M_gate.mean():.4f}, std={M_gate.std():.4f}")
print(f"  up_proj (linear output):     shape {M_up.shape}   | mean={M_up.mean():.4f}, std={M_up.std():.4f}")
print(f"  down_proj (compound input):  shape {M_down.shape} | mean={M_down.mean():.4f}, std={M_down.std():.4f}")

Baseline Profiling & MNLI Inference: 100%|██████████| 1000/1000 [00:34<00:00, 28.64it/s]



Uncompressed Baseline Accuracy: 48.00%
Captured Activation Profiles:
  gate_proj (act_fn output):   shape (1000, 6912) | mean=0.0801, std=0.3907
  up_proj (linear output):     shape (1000, 6912)   | mean=0.0004, std=0.4217
  down_proj (compound input):  shape (1000, 6912) | mean=0.0002, std=0.1507
time: 34.99s
cummulative_time: 44.64s


## Step 2: Dedicated DBSCAN Clustering & Tensor Construction Per Submodule

We run DBSCAN independently on each submodule's representative activation values ($v_c = \text{mean}_t A_{t, c}$):
- Each submodule uses an $\epsilon$ calibrated to its own empirical variance:
  $$\epsilon = \max(0.04, 0.18 \cdot \sigma_v)$$
- Superweights (noise `-1`, $|x| > 3.0$ or top 1% variance) are isolated and quarantined in uncompressed FP32.
- The top 6 dense clusters are sub-chunked into uniform slices of size $M=400$ ($2,400$ coordinates total).
- For `gate_proj` and `up_proj`, rows are sliced into $[6, 400, 1152]$.
- For `down_proj`, columns are sliced and transposed into $[6, 400, 1152]$.

In [5]:
# =====================================================================
# STEP 5: Independent DBSCAN Clustering & 3D Tensor Construction
# =====================================================================
CHUNK_SIZE = 400
NUM_CHUNKS = 6

def cluster_submodule(name, acts_matrix, weight_tensor, is_col=False):
    v = np.mean(acts_matrix, axis=0)
    std_v = np.std(v)
    eps = max(0.04, float(std_v * 0.18))
    
    db = DBSCAN(eps=eps, min_samples=30, metric="euclidean")
    labels = db.fit_predict(v.reshape(-1, 1))
    
    max_mags = np.max(np.abs(acts_matrix), axis=0)
    variances = np.var(acts_matrix, axis=0)
    super_mask = (labels == -1) | (max_mags > 3.0) | (variances >= np.quantile(variances, 0.99))
    super_indices = np.where(super_mask)[0]
    
    unique_labels = [lab for lab in np.unique(labels) if lab != -1]
    
    chunk_list = []
    for lab in unique_labels:
        c_idx = np.where((labels == lab) & (~super_mask))[0]
        if len(c_idx) == 0:
            continue
        sorted_idx = c_idx[np.argsort(v[c_idx])]
        num_full = len(sorted_idx) // CHUNK_SIZE
        for ci in range(num_full):
            chunk_list.append(sorted_idx[ci * CHUNK_SIZE : (ci + 1) * CHUNK_SIZE])
            if len(chunk_list) >= NUM_CHUNKS:
                break
        if len(chunk_list) >= NUM_CHUNKS:
            break
            
    if len(chunk_list) < NUM_CHUNKS:
        assigned = set(np.concatenate(chunk_list) if chunk_list else [])
        avail = [i for i in range(len(v)) if i not in assigned and not super_mask[i]]
        needed = NUM_CHUNKS - len(chunk_list)
        for ci in range(needed):
            if len(avail) >= CHUNK_SIZE:
                chunk_list.append(np.array(avail[:CHUNK_SIZE]))
                avail = avail[CHUNK_SIZE:]
                
    active_coords = np.concatenate(chunk_list)
    
    if is_col:
        # Columns sliced and transposed to [6, 400, 1152]
        T = torch.stack([weight_tensor[:, c].T.float().cpu() for c in chunk_list], dim=0)
    else:
        # Rows sliced into [6, 400, 1152]
        T = torch.stack([weight_tensor[c, :].float().cpu() for c in chunk_list], dim=0)
        
    print(f"Submodule {name:<10}: eps={eps:.4f} | Dense Clusters={len(unique_labels)} | "
          f"Superweights={len(super_indices)} ({len(super_indices)/len(v)*100:.1f}%) | Tensor={list(T.shape)}")
    
    return {
        "name": name,
        "tensor": T,
        "chunk_list": chunk_list,
        "active_coords": active_coords,
        "super_indices": super_indices,
        "is_col": is_col,
    }

submodule_data = {
    "gate_proj": cluster_submodule("gate_proj", M_gate, W_gate_orig, is_col=False),
    "up_proj":   cluster_submodule("up_proj",   M_up,   W_up_orig,   is_col=False),
    "down_proj": cluster_submodule("down_proj", M_down, W_down_orig, is_col=True),
}

Submodule gate_proj : eps=0.0700 | Dense Clusters=2 | Superweights=203 (2.9%) | Tensor=[6, 400, 1152]
Submodule up_proj   : eps=0.0737 | Dense Clusters=1 | Superweights=203 (2.9%) | Tensor=[6, 400, 1152]
Submodule down_proj : eps=0.0400 | Dense Clusters=1 | Superweights=143 (2.1%) | Tensor=[6, 400, 1152]
time: 0.72s
cummulative_time: 45.37s


## Step 3: Multi-Tier Sweep Across All 3 Submodules

We evaluate 4 distinct compression strategies to map the accuracy-compression landscape:
1. **Moderate Sweet Spot (`[4, 180, 600]` on all 3)**:
   Targets ~76–78% reconstruction error across all submodules.
2. **Balanced Tier (`[4, 250, 800]` on all 3)**:
   Targets ~68–70% reconstruction error across all submodules.
3. **Mixed Optimal Tier (Heterogeneous)**:
   - `gate_proj`: `[3, 100, 350]` (aggressive cut + regularizing denoising)
   - `up_proj`: `[4, 180, 600]` (moderate high-fidelity cut)
   - `down_proj`: `[4, 180, 600]` (moderate high-fidelity cut)
4. **Aggressive Tier (`[3, 100, 350]` on all 3)**:
   Tests aggressive cut with independent clustering to measure gain over coupled clustering.

In [6]:
# =====================================================================
# STEP 6: Multi-Tier Optimization, Live Injection & MNLI Evaluation
# =====================================================================
def optimize_tucker_gd(T, ranks, num_steps=35, lr=1e-3, device="cpu"):
    core_init, factors_init = tucker(T, rank=ranks, init='svd')
    core_param = torch.nn.Parameter(core_init.clone().to(device))
    factors_param = [torch.nn.Parameter(f.clone().to(device)) for f in factors_init]
    optimizer = torch.optim.Adam([core_param] + factors_param, lr=lr)
    T_target = T.to(device)

    for step in range(num_steps):
        optimizer.zero_grad()
        T_recon = tucker_to_tensor((core_param, factors_param))
        loss = torch.norm(T_target - T_recon) ** 2
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        T_recon_final = tucker_to_tensor((core_param, factors_param)).cpu()
        final_err = (torch.norm(T.cpu() - T_recon_final) / torch.norm(T.cpu())).item()

    return core_param.detach().cpu(), [f.detach().cpu() for f in factors_param], T_recon_final, final_err

eval_tiers = [
    {
        "name": "Moderate (77% Sweet Spot)",
        "ranks": {
            "gate_proj": [4, 180, 600],
            "up_proj":   [4, 180, 600],
            "down_proj": [4, 180, 600],
        }
    },
    {
        "name": "Balanced Tier",
        "ranks": {
            "gate_proj": [4, 250, 800],
            "up_proj":   [4, 250, 800],
            "down_proj": [4, 250, 800],
        }
    },
    {
        "name": "Mixed Optimal (Aggressive Gate + Moderate Up/Down)",
        "ranks": {
            "gate_proj": [3, 100, 350],
            "up_proj":   [4, 180, 600],
            "down_proj": [4, 180, 600],
        }
    },
    {
        "name": "Aggressive Tier (All 3 Submodules)",
        "ranks": {
            "gate_proj": [3, 100, 350],
            "up_proj":   [3, 100, 350],
            "down_proj": [3, 100, 350],
        }
    },
]

tier_results = []

for tier in eval_tiers:
    tier_name = tier["name"]
    tier_ranks = tier["ranks"]
    
    print(f"\n{'='*75}")
    print(f"Evaluating Tier: {tier_name}")
    print(f"{'='*75}")
    
    # 1. Optimize and inject each submodule
    sub_errors = {}
    sub_params_saved = 0
    
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        sdata = submodule_data[sub_name]
        ranks = tier_ranks[sub_name]
        
        cg, fg, T_recon, err = optimize_tucker_gd(sdata["tensor"], ranks, num_steps=35, lr=1e-3, device="cpu")
        sub_errors[sub_name] = err
        
        # Parameter Accounting
        orig_p = sdata["tensor"].numel()
        comp_p = cg.numel() + sum(f.numel() for f in fg)
        sub_params_saved += (orig_p - comp_p)
        
        # Live Injection
        mod_ref = getattr(target_layer.mlp, sub_name)
        orig_w = (W_gate_orig if sub_name == "gate_proj" else W_up_orig if sub_name == "up_proj" else W_down_orig)
        mod_ref.weight.data = orig_w.clone().to(model.device)
        
        if sdata["is_col"]:
            # down_proj: columns
            for k, c in enumerate(sdata["chunk_list"]):
                mod_ref.weight.data[:, c] = T_recon[k].T.to(device=model.device, dtype=mod_ref.weight.dtype)
            mod_ref.weight.data[:, sdata["super_indices"]] = orig_w[:, sdata["super_indices"]].to(model.device)
        else:
            # gate_proj, up_proj: rows
            for k, c in enumerate(sdata["chunk_list"]):
                mod_ref.weight.data[c, :] = T_recon[k].to(device=model.device, dtype=mod_ref.weight.dtype)
            mod_ref.weight.data[sdata["super_indices"], :] = orig_w[sdata["super_indices"], :].to(model.device)
            
        print(f"  {sub_name:<12} (Ranks {ranks}): GD Recon Error = {err*100:.2f}% | Saved: {orig_p - comp_p:,} params")
        
    # 2. Evaluate on GLUE MNLI
    tier_preds = []
    tier_gts = []
    
    model.eval()
    with torch.no_grad():
        for sample in tqdm(eval_data, desc=f"Eval {tier_name}"):
            prompt = (
                f"<start_of_turn>user\n"
                f"Premise: {sample['premise']}\n"
                f"Hypothesis: {sample['hypothesis']}\n"
                f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
                f"Answer with one word\n"
                f"<start_of_turn>model\n"
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            outputs = model(**inputs, logits_to_keep=1)

            next_token_logits = outputs.logits[0, -1, :]
            candidate_logits = next_token_logits[label_token_ids]
            pred_label = torch.argmax(candidate_logits).item()

            tier_preds.append(pred_label)
            tier_gts.append(sample["label"])
            
    acc = accuracy_score(tier_gts, tier_preds)
    delta = acc - baseline_accuracy
    
    print(f"\nResult for {tier_name}:")
    print(f"  Accuracy:      {acc * 100:.2f}% (Δ: {delta * 100:+.2f}%)")
    print(f"  Total Cut:     {sub_params_saved:,} params in Layer 0 (3 submodules)")
    
    tier_results.append({
        "Variant": tier_name,
        "Ranks": str(tier_ranks),
        "Params_Saved": sub_params_saved,
        "Gate_Err": round(sub_errors["gate_proj"] * 100, 2),
        "Up_Err": round(sub_errors["up_proj"] * 100, 2),
        "Down_Err": round(sub_errors["down_proj"] * 100, 2),
        "Accuracy": round(acc * 100, 2),
        "Delta": round(delta * 100, 2),
    })

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Restore pristine weights to model
target_layer.mlp.gate_proj.weight.data = W_gate_orig.clone().to(model.device)
target_layer.mlp.up_proj.weight.data = W_up_orig.clone().to(model.device)
target_layer.mlp.down_proj.weight.data = W_down_orig.clone().to(model.device)
print("\nRestored Layer 0 to pristine weights.")


Evaluating Tier: Moderate (77% Sweet Spot)
  gate_proj    (Ranks [4, 180, 600]): GD Recon Error = 76.72% | Saved: 1,569,576 params
  up_proj      (Ranks [4, 180, 600]): GD Recon Error = 78.09% | Saved: 1,569,576 params
  down_proj    (Ranks [4, 180, 600]): GD Recon Error = 76.09% | Saved: 1,569,576 params


Eval Moderate (77% Sweet Spot): 100%|██████████| 1000/1000 [00:31<00:00, 31.81it/s]



Result for Moderate (77% Sweet Spot):
  Accuracy:      40.10% (Δ: -7.90%)
  Total Cut:     4,708,728 params in Layer 0 (3 submodules)

Evaluating Tier: Balanced Tier
  gate_proj    (Ranks [4, 250, 800]): GD Recon Error = 69.37% | Saved: 943,176 params
  up_proj      (Ranks [4, 250, 800]): GD Recon Error = 70.64% | Saved: 943,176 params
  down_proj    (Ranks [4, 250, 800]): GD Recon Error = 68.45% | Saved: 943,176 params


Eval Balanced Tier: 100%|██████████| 1000/1000 [00:30<00:00, 32.48it/s]



Result for Balanced Tier:
  Accuracy:      41.60% (Δ: -6.40%)
  Total Cut:     2,829,528 params in Layer 0 (3 submodules)

Evaluating Tier: Mixed Optimal (Aggressive Gate + Moderate Up/Down)
  gate_proj    (Ranks [3, 100, 350]): GD Recon Error = 88.58% | Saved: 2,216,582 params
  up_proj      (Ranks [4, 180, 600]): GD Recon Error = 78.09% | Saved: 1,569,576 params
  down_proj    (Ranks [4, 180, 600]): GD Recon Error = 76.09% | Saved: 1,569,576 params


Eval Mixed Optimal (Aggressive Gate + Moderate Up/Down): 100%|██████████| 1000/1000 [00:30<00:00, 32.86it/s]



Result for Mixed Optimal (Aggressive Gate + Moderate Up/Down):
  Accuracy:      37.90% (Δ: -10.10%)
  Total Cut:     5,355,734 params in Layer 0 (3 submodules)

Evaluating Tier: Aggressive Tier (All 3 Submodules)
  gate_proj    (Ranks [3, 100, 350]): GD Recon Error = 88.58% | Saved: 2,216,582 params
  up_proj      (Ranks [3, 100, 350]): GD Recon Error = 89.77% | Saved: 2,216,582 params
  down_proj    (Ranks [3, 100, 350]): GD Recon Error = 88.25% | Saved: 2,216,582 params


Eval Aggressive Tier (All 3 Submodules): 100%|██████████| 1000/1000 [00:30<00:00, 33.00it/s]


Result for Aggressive Tier (All 3 Submodules):
  Accuracy:      39.50% (Δ: -8.50%)
  Total Cut:     6,649,746 params in Layer 0 (3 submodules)

Restored Layer 0 to pristine weights.
time: 150.04s
cummulative_time: 195.42s


## Step 4: Summary Table & Artifact Export

We tabulate the comparative results across all tiers and export metrics to `artifacts/08_independent_sublayer_results.json`.

In [7]:
# =====================================================================
# STEP 7: Tabulate Comparative Results & Export Metrics
# =====================================================================
print("=" * 110)
print(f"{'Variant':<42} | {'Gate Err':<9} | {'Up Err':<8} | {'Down Err':<9} | {'Params Cut':<11} | {'Accuracy':<9} | {'Delta':<8}")
print("=" * 110)
print(f"{'Baseline (Uncompressed)':<42} | {'0.00%':<9} | {'0.00%':<8} | {'0.00%':<9} | {'0':<11} | {baseline_accuracy*100:>7.2f}% | {'+0.00%':<8}")

for r in tier_results:
    print(f"{r['Variant']:<42} | {r['Gate_Err']:>6.2f}% | {r['Up_Err']:>5.2f}% | {r['Down_Err']:>6.2f}% | {r['Params_Saved']:>10,} | {r['Accuracy']:>7.2f}% | {r['Delta']:>+6.2f}%")

print("=" * 110)

os.makedirs("artifacts", exist_ok=True)
output_payload = {
    "experiment": "08_independent_sublayer_dbscan_tucker",
    "target_model": model_id,
    "layer": 0,
    "baseline_accuracy": round(baseline_accuracy * 100, 2),
    "tiers": tier_results,
    "timings": NOTEBOOK_TIMINGS,
}

with open("artifacts/08_independent_sublayer_results.json", "w") as f:
    json.dump(output_payload, f, indent=2)

print(f"Saved benchmark results to artifacts/08_independent_sublayer_results.json")

Variant                                    | Gate Err  | Up Err   | Down Err  | Params Cut  | Accuracy  | Delta   
Baseline (Uncompressed)                    | 0.00%     | 0.00%    | 0.00%     | 0           |   48.00% | +0.00%  
Moderate (77% Sweet Spot)                  |  76.72% | 78.09% |  76.09% |  4,708,728 |   40.10% |  -7.90%
Balanced Tier                              |  69.37% | 70.64% |  68.45% |  2,829,528 |   41.60% |  -6.40%
Mixed Optimal (Aggressive Gate + Moderate Up/Down) |  88.58% | 78.09% |  76.09% |  5,355,734 |   37.90% | -10.10%
Aggressive Tier (All 3 Submodules)         |  88.58% | 89.77% |  88.25% |  6,649,746 |   39.50% |  -8.50%
Saved benchmark results to artifacts/08_independent_sublayer_results.json
time: 0.00s
cummulative_time: 195.42s
